# Saligue AI Studio — Saligue by INDECIANA

Runs Saligue's own image AI (Leffa + SDXL Inpainting, open models) on a free Google Colab GPU and gives you a **test link**.

**How to use**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**. First run takes about 10–15 minutes (downloads ~20 GB of models).
3. At the bottom you get two links:
   - **AI Studio** — test the AI directly in the browser.
   - **Saligue app with realistic AI** — open on your phone or computer.

The links work while this page stays open (Colab free sessions last a few hours).

In [ ]:
# 1. Check that a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: choose Runtime > Change runtime type > T4 GPU"

In [ ]:
# 2. Get Saligue AI Studio and install what it needs
!git clone --depth 1 https://github.com/ColFelixHabibi/SaligueProject.git /content/SaligueProject 2>/dev/null || git -C /content/SaligueProject pull
%cd /content/SaligueProject/ai-server
!pip install -q -r requirements.txt

In [ ]:
# 3. Download the open models (Leffa code + weights, SDXL Inpainting)
!python setup_models.py

In [ ]:
# 4. Start the AI server in the background and wait until it is ready
import subprocess, time, urllib.request
server = subprocess.Popen(["python", "server.py"], stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        print(urllib.request.urlopen("http://localhost:7860/api/health").read().decode())
        break
    except Exception:
        time.sleep(5)
else:
    print(open("server.log").read()[-3000:])
    raise SystemExit("Server did not start; see the log above.")

In [ ]:
# 5. Open a free public link (Cloudflare quick tunnel, no account needed)
import re, subprocess, time, urllib.parse
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:7860", "--no-autoupdate"], stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    found = re.search(r"https://[a-z0-9-]+.trycloudflare.com", open("tunnel.log").read())
    if found:
        url = found.group(0)
        break
    time.sleep(2)
if not url:
    raise SystemExit("Could not open the public link; run this cell again.")
time.sleep(5)
app_link = "https://colfelixhabibi.github.io/SaligueProject/mirror/?ai=" + urllib.parse.quote(url, safe="")
from IPython.display import HTML, display
display(HTML(f"""
<h2 style="color:#e21376">Saligue AI Studio is running</h2>
<p><b>AI Studio (test in browser):</b> <a href="{url}/studio/" target="_blank">{url}/studio/</a></p>
<p><b>Saligue app with realistic AI:</b> <a href="{app_link}" target="_blank">{app_link}</a></p>
<p>Keep this Colab page open while testing.</p>
"""))

In [ ]:
# Server log (run this cell if something goes wrong)
print(open("/content/SaligueProject/ai-server/server.log").read()[-5000:])